# 🤗 x 🦾: Training ACT with LeRobot Notebook

Welcome to the **LeRobot ACT training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `ACT` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `ACT` policy for 100,000 steps typically takes **about 1.5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer.

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


In [12]:
# Import Colab Secrets userdata module
from google.colab import userdata

# Set OpenAI API key
import os
os.environ["HUGGINGFACE_TOKEN"] = userdata.get('HUGGINGFACE_TOKEN')

# Set other API keys similarly
os.environ["WANDB_TOKEN"] = userdata.get('WANDB_TOKEN')

# Grab Google Colab secrets
Make `HUGGINGFACE_TOKEN` and `WANDB_TOKEN` from Colab secrets into env varables.

## Install conda
This cell uses `condacolab` to bootstrap a full Conda environment inside Google Colab.


In [5]:
#!pip install -q condacolab
#import condacolab
#condacolab.install()

# Installing another way with a python 3.12 version
# 1. Download the installer script
!wget https://repo.anaconda.com/miniconda/Miniconda3-py312_26.3.2-2-Linux-x86_64.sh

# 2. Make the script executable
!chmod +x Miniconda3-py312_26.3.2-2-Linux-x86_64.sh

# 3. Run the installer in batch mode to /usr/local
!bash ./Miniconda3-py312_26.3.2-2-Linux-x86_64.sh -b -f -p /usr/local

# 4. Add the new path to sys.path so you can import libraries
import sys
sys.path.append('/usr/local/lib/python3.12/site-packages/')

--2026-05-14 02:36:56--  https://repo.anaconda.com/miniconda/Miniconda3-py312_26.3.2-2-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 161366816 (154M) [application/octet-stream]
Saving to: ‘Miniconda3-py312_26.3.2-2-Linux-x86_64.sh’

Miniconda3-py312_26 100%[===================>] 153.89M   201MB/s    in 0.8s    

2026-05-14 02:36:57 (201 MB/s) - ‘Miniconda3-py312_26.3.2-2-Linux-x86_64.sh’ saved [161366816/161366816]

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    F

## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg (version 7.1.1), and installs the package in editable mode.


In [6]:
!git clone https://github.com/huggingface/lerobot.git
!conda install ffmpeg=7.1.1 -c conda-forge
!cd lerobot && pip install -e .

Cloning into 'lerobot'...
remote: Enumerating objects: 50422, done.
remote: Counting objects: 100% (470/470), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 50422 (delta 387), reused 291 (delta 291), pack-reused 49952 (from 3)
Receiving objects: 100% (50422/50422), 241.38 MiB | 29.98 MiB/s, done.
Resolving deltas: 100% (32125/32125), done.
Jupyter detected...

CondaToSNonInteractiveError: Terms of Service have not been accepted for the following channels. Please accept or remove them before proceeding:
    - https://repo.anaconda.com/pkgs/main
    - https://repo.anaconda.com/pkgs/r

To accept these channels' Terms of Service, run the following commands:
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

For information on safely removing channels from your conda configuration,
please see the official documentation:

    https://www.anaconda.co

## Weights & Biases login
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging.

In [10]:
!pip install wandb
!wandb login ${WANDB}

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 

Aborted!


## Start training ACT with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--policy.type=act`:  
   Specifies the policy configuration to use. `act` refers to [configuration_act.py](../lerobot/common/policies/act/configuration_act.py), which will automatically adapt to your dataset’s setup (e.g., number of motors and cameras).

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this. Set to `False` if you do not plan on using Weights & Biases.

In [13]:
!cd lerobot && python src/lerobot/scripts/lerobot_train.py \
  --dataset.repo_id=dataset.repo_id=rcpeters0/putBlueLegoAwayV2 \
  --policy.type=act \
  --output_dir=outputs/train/bluelego \
  --job_name=bluelego_training_job \
  --policy.device=cuda \
  --wandb.enable=False \
  --policy.repo_id=rcpeters0/bluelego_act_recordpolicy0

Traceback (most recent call last):
  File "/content/lerobot/src/lerobot/scripts/lerobot_train.py", line 46, in <module>
    from lerobot.datasets import EpisodeAwareSampler, make_dataset
  File "/content/lerobot/src/lerobot/datasets/__init__.py", line 20, in <module>
    require_package("datasets", extra="dataset")
  File "/content/lerobot/src/lerobot/utils/import_utils.py", line 92, in require_package
    raise ImportError(
ImportError: 'datasets' is required but not installed. Install it with: pip install 'lerobot[dataset]' (or uv pip install 'lerobot[dataset]')


## Login into Hugging Face Hub.
### Now after training is done login into the Hugging Face hub and upload the last checkpoint.

In [ ]:
from huggingface_hub import HfApi

HF_USERNAME = "${HF_USER}"
HF_REPO_NAME = "act-configs"

api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
files_in_repo = api.list_repo_files(repo_id=repo_id)

print(f"Files in {repo_id}:")
for file in files_in_repo:
    print(f"- {file}")


### Configure Hugging Face Token

Add your HF_TOKEN (AKA Secret) to Google Colab to enable Colab to access your HF repositories. This is optional and might need modification if you are using another cloud provider.

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Verify token is loaded (optional)
if os.getenv("HF_TOKEN"):
    print("Hugging Face token loaded successfully.")
else:
    print("Error: Hugging Face token not found. Please set it as a Colab secret named 'HF_TOKEN'.")

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

### Download files from Hugging Face Hub into Colab

Now, you can use `hf_hub_download` to pull the files directly to your Colab environment. You can then download them from Colab to your local machine. This is optional.

In [ ]:
from huggingface_hub import hf_hub_download

# Your Hugging Face repository details
HF_CONFIG_REPO_ID = "${HF_USER}/act-configs" # Where train_config.json should be
HF_POLICY_REPO_ID = "${HF_USER}/hf_act_recordpolicy0" # Where the trained model is

# Define the files to download
config_file_name = "train_config.json"
model_file_name = "model.safetensors"
tokenizer_processor_file_name = "tokenizer_processor.safetensors"

# Download train_config.json
train_config_path = hf_hub_download(repo_id=HF_CONFIG_REPO_ID, filename=config_file_name)
print(f"Downloaded {config_file_name} to: {train_config_path}")

# Download the trained model files
model_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=model_file_name)
print(f"Downloaded {model_file_name} to: {model_path}")

tokenizer_processor_path = hf_hub_download(repo_id=HF_POLICY_REPO_ID, filename=tokenizer_processor_file_name)
print(f"Downloaded {tokenizer_processor_file_name} to: {tokenizer_processor_path}")

print("\nAll specified files have been downloaded to your Colab environment.")
print("You can find them in the paths printed above. To download them to your local machine, right-click on the files in the Colab file browser (left sidebar) and select 'Download'.")

### Verify `train_config.json` existence locally. This is needed for restarting training and testing of the policy.

In [ ]:
!ls -l /content/lerobot/outputs/train/hf_act_record0/

In [ ]:
HF_USERNAME = "${HF_USER}"
HF_REPO_NAME = "act-configs"

!hf repo-files $HF_USERNAME/$HF_REPO_NAME

In [ ]:
!hf auth login

In [ ]:
!hf upload ${HF_USER}/hf_act_record0 \
  /content/lerobot/outputs/train/hf_act_record0/checkpoints/last/pretrained_model

In [ ]:
!hf auth login

### Create a new repository on Hugging Face Hub

This command will create a new repository under your Hugging Face account.

In [ ]:
HF_USERNAME = "${HF_USER}"
HF_REPO_NAME = "act-configs"

!hf repo create $HF_REPO_NAME --type model --private --organization $HF_USERNAME

In [ ]:
### Upload `train_config.json` to the new repository
HF_USERNAME = "${HF_USER}"
HF_REPO_NAME = "act-configs"
LOCAL_CONFIG_PATH = "/content/lerobot/outputs/train/hf_act_record0/train_config.json"

!hf upload $HF_USERNAME/$HF_REPO_NAME "$LOCAL_CONFIG_PATH" train_config.json